## Ran Uram 206661886 
## Shahar Lankry 322659137 
## Daniel Geron 212515522
-------------------------

## Imports

In [3]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
from pathlib import Path
import argparse
import pandas as pd
import re
from typing import Tuple
from tqdm import tqdm
from openpyxl import load_workbook

## Change paths to match your folders location

In [ ]:
og_images_folder=r"C:/Users/ranor/Desktop/code/Data"
normalised_images_folder=r"C:/Users/ranor/Desktop/code/Data_normalized"
feature_tables_location=r"C:/Users/ranor/Desktop/code/tables"

## Image normalization

In [5]:
def binarize_image(img: np.ndarray, method: str = 'sauvola') -> np.ndarray:
    # Converts grayscale image to binary (black and white) using thresholding.
    # This separates the ink (text) from the paper (background).

    if method == 'otsu':
        # Otsu finds a single global threshold for the whole image (good for uniform lighting)
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    elif method == 'adaptive':
        # Adaptive calculates threshold locally for small regions (better for shadows/uneven lighting)
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 21, 10
        )

    elif method == 'sauvola':
        # Sauvola is a specific adaptive method optimized for documents.
        # Here we use a tuned Adaptive Threshold as a robust fallback implementation.
        binary = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 25, 15
        )

    else:
        raise ValueError(f"Unknown binarization method: {method}")

    return binary


In [6]:
def remove_noise(binary_img: np.ndarray, min_component_size: int = 10) -> np.ndarray:
    # Cleans the image by removing small specks and dots that aren't part of the text.
    
    # Morphological Opening: Erodes then Dilates to remove tiny noise (salt-and-pepper noise)
    kernel = np.ones((2, 2), np.uint8)
    opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel, iterations=1)

    # Connected Components: Identifies all separated blobs in the image
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(255 - opened, connectivity=8)
    # Create a clean black canvas
    output = np.zeros_like(opened)

    # Copy only the components that are large enough (actual letters) to the new image
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_component_size:
            output[labels == i] = 255

    # Invert back to ensure text is black on white background
    output = 255 - output

    return output


In [7]:
def crop_to_content(img: np.ndarray, padding: int = 20) -> np.ndarray:
    # Cuts away the empty white margins, keeping only the area with actual handwriting.
    
    # Find coordinates of all black pixels (text)
    coords = cv2.findNonZero(255 - img)

    if coords is None:
        return img

    # Get the bounding box (rectangle) that surrounds all the text
    x, y, w, h = cv2.boundingRect(coords)

    # Add some padding (white space) around the text so it doesn't touch the edges
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(img.shape[1] - x, w + 2 * padding)
    h = min(img.shape[0] - y, h + 2 * padding)

    # Perform the actual crop
    cropped = img[y:y+h, x:x+w]

    return cropped

In [8]:
def normalize_height(img: np.ndarray, target_height: int = 64) -> np.ndarray:
    # Resizes the image to a fixed height (e.g., 64px) while keeping the Aspect Ratio.
    # This ensures the letters don't get squashed or stretched unnaturally.
    
    h, w = img.shape[:2]

    # Calculate new width based on aspect ratio
    aspect_ratio = w / h
    new_width = int(target_height * aspect_ratio)
    
    # Resize using INTER_AREA interpolation (best for shrinking images)
    resized = cv2.resize(img, (new_width, target_height), interpolation=cv2.INTER_AREA)

    return resized

In [9]:
def process_single_image(
    input_path: str, output_path: str, binarize: bool = True,
    denoise: bool = True, crop: bool = True, normalize_size: bool = False,
    target_height: int = 64, binarization_method: str = 'adaptive', min_noise_size: int = 10
) -> bool:
    # The main pipeline for a single file. Runs all steps in order.
    
    try:
        img = cv2.imread(input_path)
        if img is None:
            return False

        # Convert to grayscale first
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img

        processed = gray.copy()

        # Step 1: Binarization (Thresholding)
        if binarize:
            processed = binarize_image(processed, method=binarization_method)

        # Step 2: Noise Removal (Cleaning)
        if denoise:
            processed = remove_noise(processed, min_component_size=min_noise_size)

        # Step 3: Smart Cropping
        if crop:
            processed = crop_to_content(processed, padding=20)

        # Step 4: Height Normalization (Optional)
        if normalize_size and target_height > 0:
            processed = normalize_height(processed, target_height=target_height)

        # Save the final processed image
        cv2.imwrite(output_path, processed)

        return True

    except Exception as e:
        print(f"Error: {e}")
        return False


In [10]:
def process_directory(input_dir: str, output_dir: str, **kwargs) -> None:
    # Iterates over the entire folder and processes every image found.
    
    os.makedirs(output_dir, exist_ok=True)

    image_files = []
    # Find all image files with common extensions
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff']:
        image_files.extend(Path(input_dir).glob(f'*{ext}'))
        image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))

    # Sort and remove duplicates
    image_files = sorted(list(set(image_files)))
    total = len(image_files)

    print(f"Found {total} images to process")
    print(f"Output directory: {output_dir}")
    print("-" * 60)

    success_count = 0

    # Loop through all images
    for idx, img_path in enumerate(image_files, 1):
        output_path = os.path.join(output_dir, img_path.name)

        # Process the current image
        success = process_single_image(str(img_path), output_path, **kwargs)

        if success:
            success_count += 1
            if idx % 100 == 0:
                print(f"Processed {idx}/{total} images...")
        else:
            print(f"[ERROR] Failed to process {img_path.name}")

    print("-" * 60)
    print(f"Done! Successfully processed: {success_count}/{total}")

In [11]:

    # 3. Processing Settings (Change to False if you want to skip a step)
should_binarize = True
should_denoise = True
should_crop = True
should_normalize_height = False
    

    # Check if input directory exists before starting
if 'og_images_folder' not in locals():
    print("Error: Variables not defined. Please run the cell with the paths first (Cell 2).")
elif not os.path.exists(og_images_folder):
    print(f"Error: The input folder does not exist at:\n{og_images_folder}")
else:
    print(f"Folder found! Processing images...")

    # Run the processing
    process_directory(
            input_dir=og_images_folder,
            output_dir=normalised_images_folder,
            binarize=should_binarize,
            denoise=should_denoise,
            crop=should_crop,
            normalize_size=should_normalize_height,
            target_height=64 
    )


Folder found! Processing images...
Found 2512 images to process
Output directory: C:/Users/ranor/Desktop/code/Data_normalized
------------------------------------------------------------
Processed 100/2512 images...
Processed 200/2512 images...
Processed 300/2512 images...
Processed 400/2512 images...
Processed 500/2512 images...
Processed 600/2512 images...
Processed 700/2512 images...
Processed 800/2512 images...
Processed 900/2512 images...
Processed 1000/2512 images...
Processed 1100/2512 images...
Processed 1200/2512 images...
Processed 1300/2512 images...
Processed 1400/2512 images...
Processed 1500/2512 images...
Processed 1600/2512 images...
Processed 1700/2512 images...
Processed 1800/2512 images...
Processed 1900/2512 images...
Processed 2000/2512 images...
Processed 2100/2512 images...
Processed 2200/2512 images...
Processed 2300/2512 images...
Processed 2400/2512 images...
Processed 2500/2512 images...
------------------------------------------------------------
Done! Succe

## FEATURE EXTRACTION

מילון פיצ'רים וערכי סקאלה
| שם הפיצ'ר | מה זה אומר? (תיאור) | ערכי הקיצון והאמצע (סקאלה 0-1) |
| :--- | :--- | :--- |
| נטייה (Slant) | זווית הכתיבה ביחס לאנך | 0.0 = שמאלית חזקה (נגד הכיוון)<br>0.5 = אנכי לגמרי (90 מעלות, ישר)<br>1.0 = ימנית חזקה (שוכב קדימה) |
| עובי קו (Stroke Thickness) | עובי הקו ביחס לגודל האות ("לחץ") | 0.0 = דקיק (קו נימי, עדין מאוד)<br>0.5 = בינוני (עובי סטנדרטי)<br>1.0 = עבה מאוד (קו "בצקי", מרוח) |
| היצמדות לשורה (Baseline) | מיקום האות ביחס לפס המודפס | 0.0 = חותך/יורד (הכתיבה יורדת מתחת לקו)<br>0.5 = על הקו בדיוק (התאמה מושלמת)<br>1.0 = מרחף גבוה (הכתיבה מנותקת מהקו כלפי מעלה) |

### Baseline 

In [12]:
def extract_file_id(filename: str) -> int:
    # Helper function to extract the first number from a filename for natural sorting
    match = re.search(r'(\d+)', filename)
    if match:
        return int(match.group(1))
    return -1

In [ ]:
def calculate_position_score(img: np.ndarray) -> float:
    # core logic: Calculates vertical position relative to the baseline.
    # Returns a score between 0.0 (Cutting the line) and 1.0 (Floating above).
    
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Otsu's thresholding to create binary image
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    img_h, img_w = thresh.shape

    # Define a wide horizontal kernel to detect only the ruled lines (ignoring text)
    min_line_width = int(img_w * 0.3)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
    
    # Morphological opening to isolate horizontal lines
    detected_lines_map = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    
    cnts_lines, _ = cv2.findContours(detected_lines_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not cnts_lines:
        return None

    valid_lines = []
    for c in cnts_lines:
        x, y, w, h = cv2.boundingRect(c)
        # Filter out lines that are too close to the top/bottom borders (noise)
        if y > 10 and y < img_h - 10:
            valid_lines.append(c)
            
    if not valid_lines:
        valid_lines = [max(cnts_lines, key=cv2.contourArea)]

    # Subtract the detected lines from original binary to get only the handwriting
    text_only = cv2.subtract(thresh, detected_lines_map)
    
    # Clean small noise
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    text_only = cv2.morphologyEx(text_only, cv2.MORPH_OPEN, kernel_clean)
    
    cnts_text, _ = cv2.findContours(text_only, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    text_bottoms = []
    text_centers_y = []
    
    for c in cnts_text:
        if cv2.contourArea(c) < 15:
            continue
            
        tx, ty, tw, th = cv2.boundingRect(c)
        text_bottoms.append(ty + th)
        text_centers_y.append(ty + th/2)

    if not text_bottoms:
        return 1.0

    # Use Median to determine text baseline position, ignoring outliers like descending letters ('ן', 'ך')
    median_text_bottom = np.median(text_bottoms)
    avg_text_y = np.mean(text_centers_y)

    best_line_y = 0
    min_dist_to_line = 99999
    
    # Identify which ruled line the text belongs to (closest line to text center)
    for line_c in valid_lines:
        lx, ly, lw, lh = cv2.boundingRect(line_c)
        current_line_y = ly 
        
        dist = abs(current_line_y - avg_text_y)
        if dist < min_dist_to_line:
            min_dist_to_line = dist
            best_line_y = current_line_y

    # Calculate distance: Positive = Above line, Negative = Below line
    distance = best_line_y - median_text_bottom
    
    # Sensitivity factor: determines the pixel range that maps to the 0-1 score
    SENSITIVITY = 60.0 
    
    # Normalize score: 0.5 represents text sitting exactly on the line
    raw_score = 0.5 + (distance / SENSITIVITY)
    
    # Clip result to strict 0.0 - 1.0 range
    final_score = max(0.0, min(1.0, raw_score))
    return final_score

In [14]:
def process_images_for_position(
    input_dir: str,
    output_excel: str,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:
    # Main batch processing function: Iterates images, sorts them, and saves results to Excel
    
    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    # Sort using natural sort order (e.g., 1, 2, ... 10 instead of 1, 10, 2)
    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    
    total = len(image_files)

    print(f"Found {total} images to analyze")
    print(f"Output Excel file: {output_excel}")
    
    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            file_id = extract_file_id(img_path.name)
            img = cv2.imread(str(img_path))

            if img is None:
                print(f"[WARNING] Could not read: {img_path.name}")
                results.append({
                    'ID_Number': file_id,
                    'Filename': img_path.name,
                    'value': None
                })
                continue

            score = calculate_position_score(img)
            
            # Fallback to neutral score (0.5) if detection fails
            if score is None:
                score = 0.5
                print(f"[INFO] Detection fallback for {img_path.name}")

            results.append({
                'ID_Number': file_id,
                'Filename': img_path.name,
                'value': round(score, 3)
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] Failed to process {img_path.name}: {str(e)}")
            results.append({
                'ID_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'value': None
            })

    print(f"Analysis complete!")

    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Line Position')

    print(f"\nExcel file saved: {output_excel}")
    print("\nPosition Statistics (0=Cutting, 0.5=On Line, 1=Floating):")
    print(f"Min: {df['value'].min():.3f}")
    print(f"Max: {df['value'].max():.3f}")
    print(f"Mean: {df['value'].mean():.3f}")
    print(f"Median: {df['value'].median():.3f}")
    print(f"Std Dev: {df['value'].std():.3f}")

    return df


In [15]:
process_images_for_position(input_dir=normalised_images_folder, output_excel=os.path.join(feature_tables_location,"Baseline.xlsx"))

Found 2512 images to analyze
Output Excel file: C:/Users/ranor/Desktop/code/tables\Baseline.xlsx
[ERROR] Failed to process txt (1).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (2).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (3).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (4).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (5).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (6).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (7).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (8).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (9).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (10).png: 'numpy.ndarray' object has no attribute 'get'
[ERROR] Failed to process txt (11).png: 'numpy.ndarr

,ID_Number,Filename,value
0,1,txt (1).png,None
1,2,txt (2).png,None
2,3,txt (3).png,None
3,4,txt (4).png,None
4,5,txt (5).png,None
...,...,...,...
2507,2520,txt (2520).png,None
2508,2521,txt (2521).png,None
2509,2522,txt (2522).png,None
2510,2523,txt (2523).png,None


### Slant

In [ ]:
def measure_slant_by_shear(binary_img: np.ndarray) -> Tuple[float, float]:
    # Calculates the slant by applying affine shear transformations to the image.
    # Logic: We shear the image at different angles. The angle that maximizes the 
    # variance of the vertical projection is the one that makes the text most upright.
    
    img_height, img_width = binary_img.shape
    
    # Test angles from -30 to +30 degrees
    angles_to_test = np.linspace(-30, 30, 61)
    max_variance = 0
    optimal_angle = 0

    for angle in angles_to_test:
        angle_rad = np.radians(angle)
        shear_factor = np.tan(angle_rad)
        
        # Define the Affine Transformation Matrix for shearing
        M = np.array([[1, shear_factor, 0],
                      [0, 1, 0]], dtype=np.float32)
        
        # Apply the shear transformation
        sheared = cv2.warpAffine(binary_img, M, (img_width + abs(int(shear_factor * img_height)), img_height),
                                flags=cv2.INTER_LINEAR, borderValue=255)
        
        # Calculate vertical projection (sum of black pixels per column)
        projection = np.sum(sheared == 0, axis=0)
        
        # Calculate variance - higher variance means sharper peaks/valleys, indicating upright text
        variance = np.var(projection)

        if variance > max_variance:
            max_variance = variance
            optimal_angle = angle

    # Normalize the result to 0.0 - 1.0 range
    # -30 deg = 0.0, 0 deg = 0.5, +30 deg = 1.0
    slant = (optimal_angle + 30) / 60
    slant = np.clip(slant, 0.0, 1.0)
    
    return slant, optimal_angle

In [ ]:
def measure_slant_by_moments(binary_img: np.ndarray) -> float:
    # Calculates slant by finding contours of individual letters and fitting ellipses.
    # The average angle of these ellipses represents the handwriting slant.
    
    # Invert image to find contours (OpenCV expects white object on black background)
    inverted = 255 - binary_img
    contours, _ = cv2.findContours(inverted, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return 0.5

    angles = []
    img_area = binary_img.shape[0] * binary_img.shape[1]

    for contour in contours:
        area = cv2.contourArea(contour)
        
        # Filter noise: Ignore contours that are too small or too large
        if area < img_area * 0.0001 or area > img_area * 0.15:
            continue
        if len(contour) < 5:
            continue

        try:
            # Fit an ellipse to the contour to determine orientation
            (x, y), (MA, ma), angle = cv2.fitEllipse(contour)
            
            # Adjust angle to be relative to vertical axis
            if angle > 135:
                angle = angle - 180
            elif angle > 45:
                angle = 90 - angle
            angles.append(angle)
        except:
            continue

    if not angles:
        return 0.5

    # Remove statistical outliers using Interquartile Range (IQR) to get a robust mean
    angles = np.array(angles)
    q1, q3 = np.percentile(angles, [25, 75])
    iqr = q3 - q1
    if iqr > 0:
        mask = (angles >= q1 - 1.5*iqr) & (angles <= q3 + 1.5*iqr)
        angles = angles[mask]

    if len(angles) == 0:
        return 0.5

    mean_angle = np.mean(angles)
    
    # Normalize to 0.0 - 1.0 range based on -20 to +20 degrees limits
    mean_angle = np.clip(mean_angle, -20, 20)
    slant = (mean_angle + 20) / 40
    return slant

In [ ]:
def extract_slant(image_path: str) -> Tuple[float, dict]:
    # Main function to process a single image.
    # Combines Shear method (global) and Moments method (local) for better accuracy.
    
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.5, {'error': 'Failed to read'}

    # Ensure binary image (Thresholding) if not already
    if len(np.unique(img)) > 2:
        _, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    # Run both algorithms
    slant_shear, optimal_angle = measure_slant_by_shear(img)
    slant_moments = measure_slant_by_moments(img)

    # Weighted Average: Giving more weight (0.7) to the Shear method as it is generally more robust for lines
    slant = 0.7 * slant_shear + 0.3 * slant_moments
    slant = np.clip(slant, 0.0, 1.0)

    debug_info = {
        'slant_shear': slant_shear,
        'slant_moments': slant_moments,
        'optimal_angle': optimal_angle
    }
    return slant, debug_info

In [ ]:
def process_directory(input_dir: str, output_file: str) -> pd.DataFrame:
    # Iterates over all images in the directory, calculates slant, and saves to Excel.
    
    image_files = []
    # Collect all common image formats
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff']:
        image_files.extend(Path(input_dir).glob(f'*{ext}'))
        image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(set(image_files))
    total = len(image_files)

    print(f"Found {total} images")
    print(f"Output: {output_file}")
    
    results = []
    # Process images with a progress bar
    for idx, img_path in enumerate(tqdm(image_files, desc="Processing"), 1):
        slant, debug = extract_slant(str(img_path))
        results.append({
            'Image Number': idx,
            'File Name': img_path.name,
            'Slant': round(slant, 3)
        })

    # Save initial data to Excel
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False, engine='openpyxl')

    # Post-processing: Add Excel formula to extract image index from filename if needed
    wb = load_workbook(output_file)
    ws = wb.active
    for row_num in range(2, len(results) + 2):
        ws[f'A{row_num}'] = f'=VALUE(MID(B{row_num}, FIND("(",B{row_num})+1, FIND(")",B{row_num})-FIND("(",B{row_num})-1))'
    wb.save(output_file)

    print(f"\nStatistics:")
    print(f"  Mean: {df['Slant'].mean():.3f}")
    print(f"  Median: {df['Slant'].median():.3f}")
    print(f"  Std Dev: {df['Slant'].std():.3f}")
    print(f"  Min: {df['Slant'].min():.3f}")
    print(f"  Max: {df['Slant'].max():.3f}")
    return df

In [ ]:
process_directory(normalised_images_folder,os.path.join(feature_tables_location, 'slant.xlsx'))

Found 2512 images
Output: C:/Users/ranor/Desktop/code/tables\slant.xlsx


Processing: 100%|██████████| 2512/2512 [01:24<00:00, 29.73it/s]



Statistics:
  Mean: 0.535
  Median: 0.520
  Std Dev: 0.130
  Min: 0.015
  Max: 1.000


,Image Number,File Name,Slant
0,1,txt (1).png,0.433
1,2,txt (10).png,0.589
2,3,txt (100).png,0.585
3,4,txt (1000).png,0.611
4,5,txt (1001).png,0.439
...,...,...,...
2507,2508,txt (995).png,0.466
2508,2509,txt (996).png,0.483
2509,2510,txt (997).png,0.611
2510,2511,txt (998).png,0.633


### Stroke Thickness

In [ ]:
def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    # Preprocesses image to remove horizontal ruled lines and noise, isolating handwriting
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Create binary inverse (white text on black background)
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = img.shape[:2]

    # Detect and remove horizontal lines using Hough Transform
    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi/180, threshold=100, minLineLength=width*0.3, maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw thick lines over detected ruled lines to mask them
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    # Subtract lines from the binary image
    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))
    
    # Filter out noise and irrelevant regions using Connected Components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        # Define regions to ignore (header, footer) and noise threshold
        is_bottom_region = (y + h) > height * 0.85
        is_top_region = y < height * 0.1
        is_too_small = area < 15

        # Keep only valid handwriting components
        if not (is_bottom_region or is_top_region or is_too_small):
            handwriting_mask[labels == i] = 255

    # Invert back to normal (black text on white background)
    result = cv2.bitwise_not(handwriting_mask)
    return result

In [ ]:
def calculate_stroke_thickness_pure(img: np.ndarray) -> float:
    # Calculates average stroke thickness using Distance Transform
    handwriting_only = remove_printed_text_and_lines(img)

    if len(handwriting_only.shape) == 3:
        gray = cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
    else:
        gray = handwriting_only

    # Binary thresholding
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    inverted = cv2.bitwise_not(binary)

    # Distance Transform: calculates distance of each pixel to the nearest zero pixel (background)
    # This effectively measures the "skeleton" width at every point
    dist_transform = cv2.distanceTransform(inverted, cv2.DIST_L2, 5)
    text_distances = dist_transform[dist_transform > 0]

    # Calculate ratio of handwriting pixels to total area
    total_pixels = inverted.shape[0] * inverted.shape[1]
    handwriting_ratio = len(text_distances) / total_pixels if total_pixels > 0 else 0

    # Safety check: If page is empty or has too little content, try raw image or return 0
    MINIMUM_CONTENT_THRESHOLD = 0.003 
    if len(text_distances) == 0 or handwriting_ratio < MINIMUM_CONTENT_THRESHOLD:
        if len(img.shape) == 3:
            gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray_raw = img

        _, binary_raw = cv2.threshold(gray_raw, 127, 255, cv2.THRESH_BINARY)
        inverted_raw = cv2.bitwise_not(binary_raw)

        dist_transform_raw = cv2.distanceTransform(inverted_raw, cv2.DIST_L2, 5)
        text_distances = dist_transform_raw[dist_transform_raw > 0]

        if len(text_distances) == 0:
            return 0.0
    
    # Mean distance is the radius, so multiply by 2 for diameter (thickness)
    avg_thickness = np.mean(text_distances) * 2

    return avg_thickness


In [ ]:
def process_images_for_stroke_thickness(
    input_dir: str,
    output_excel: str,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:
    # Main processing loop: iterates images, calculates raw thickness, and normalizes results

    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(list(image_files))
    total = len(image_files)

    print(f"Found {total} images to analyze")
    print(f"Output Excel file: {output_excel}")
    
    raw_thickness_values = []
    temp_results = []

    # Step 1: Calculate raw thickness for all images
    for idx, img_path in enumerate(image_files, 1):
        try:
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"[WARNING] Could not read: {img_path.name}")
                temp_results.append({
                    'Image_Number': idx,
                    'Filename': img_path.name,
                    'Raw_Thickness': None
                })
                continue

            stroke_thickness = calculate_stroke_thickness_pure(img)
            raw_thickness_values.append(stroke_thickness)

            temp_results.append({
                'Image_Number': idx,
                'Filename': img_path.name,
                'Raw_Thickness': stroke_thickness
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] Failed to process {img_path.name}: {str(e)}")
            temp_results.append({
                'Image_Number': idx,
                'Filename': img_path.name,
                'Raw_Thickness': None
            })

    print(f"Analysis complete!")

    # Step 2: Normalize values to 0-1 range based on dataset min/max
    valid_values = [r for r in raw_thickness_values if r is not None and r > 0]
    min_thickness = min(valid_values) if valid_values else 0.0
    max_thickness = max(valid_values) if valid_values else 1.0
    thickness_range = max_thickness - min_thickness

    print(f"\nNormalizing to [0, 1] range:")
    print(f"Raw min: {min_thickness:.3f} pixels")
    print(f"Raw max: {max_thickness:.3f} pixels")

    results = []
    for item in temp_results:
        if item['Raw_Thickness'] is None:
            normalized_value = None
        elif item['Raw_Thickness'] == 0:
            normalized_value = 0.0
        elif thickness_range == 0:
            normalized_value = 0.5
        else:
            # Min-Max Normalization formula
            normalized_value = (item['Raw_Thickness'] - min_thickness) / thickness_range

        results.append({
            'Image_Number': item['Image_Number'],
            'Filename': item['Filename'],
            'Stroke_Thickness': round(normalized_value, 3)
        })
    # Save to Excel
    df = pd.DataFrame(results)
    # Ensuring the results are rounded to 3 decimal
    df['Stroke_Thickness']=df['Stroke_Thickness'].astype(float).round(3)
    df.to_excel(output_excel, index=False, sheet_name='Stroke Thickness')

    print(f"\nExcel file saved: {output_excel}")
    print("\nNormalized Stroke Thickness Statistics (0-1 range):")
    print(f"Min: {df['Stroke_Thickness'].min():.3f}")
    print(f"Max: {df['Stroke_Thickness'].max():.3f}")
    print(f"Mean: {df['Stroke_Thickness'].mean():.3f}")
    print(f"Median: {df['Stroke_Thickness'].median():.3f}")
    print(f"Std Dev: {df['Stroke_Thickness'].std():.3f}")

    return df

In [ ]:
process_images_for_stroke_thickness(input_dir=normalised_images_folder, output_excel=os.path.join(feature_tables_location, 'stroke_thickness.xlsx'))

Found 2512 images to analyze
Output Excel file: C:/Users/ranor/Desktop/code/tables\stroke_thickness.xlsx
Processed 500/2512 images...
Processed 1000/2512 images...
Processed 1500/2512 images...
Processed 2000/2512 images...
Processed 2500/2512 images...
Analysis complete!

Normalizing to [0, 1] range:
Raw min: 2.031 pixels
Raw max: 3.068 pixels

Excel file saved: C:/Users/ranor/Desktop/code/tables\stroke_thickness.xlsx

Normalized Stroke Thickness Statistics (0-1 range):
Min: 0.000
Max: 1.000
Mean: 0.294
Median: 0.290
Std Dev: 0.109


,Image_Number,Filename,Stroke_Thickness
0,1,txt (1).png,0.368
1,2,txt (10).png,0.065
2,3,txt (100).png,0.348
3,4,txt (1000).png,0.487
4,5,txt (1001).png,0.301
...,...,...,...
2507,2508,txt (995).png,0.394
2508,2509,txt (996).png,0.339
2509,2510,txt (997).png,0.191
2510,2511,txt (998).png,0.233
